
## Lesson 8: Cycles and Loops in LangGraph

### Learning objectives
- Understand cycles in graph-based workflows.
- Implement a graph that repeats tasks using conditional routing.
- Track attempts and evaluation scores through shared state.
- Define stopping conditions to prevent infinite loops.

### Core intuition

A normal graph moves forward through a fixed sequence of nodes.

A cyclic graph can return to an earlier node and repeat work.

Example: Generate Answer → Evaluate Answer → Retry or Finish.

This pattern is useful in AI agents that generate, evaluate, and improve responses.


In [1]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

In [2]:
class AgentState(TypedDict):
    answer: str
    score: int
    attempts: int

In [3]:
def generate_answer(state: AgentState):
    attempts = state["attempts"] + 1

    if attempts == 1:
        answer = "LangGraph is a framework."

    elif attempts == 2:
        answer = "LangGraph is a framework for building stateful AI workflows."

    else:
        answer = "LangGraph is a framework for building stateful AI workflows with nodes, edges, and conditional routing."

    print(f"Generation attempt: {attempts}")

    return {
        "answer": answer,
        "attempts": attempts
    }

In [4]:
def evaluate_answer(state: AgentState):
    answer = state["answer"]

    if "nodes" in answer and "edges" in answer:
        score = 90

    elif "stateful AI workflows" in answer:
        score = 70

    else:
        score = 40

    print(f"Quality score: {score}")

    return {"score": score}

In [5]:
def should_continue(state: AgentState):
    if state["score"] >= 80:
        return "done"

    if state["attempts"] >= 3:
        return "done"

    return "retry"


### Understanding the workflow

1. `generate_answer`: creates or improves the answer and increments attempts.
2. `evaluate_answer`: evaluates the answer and updates its score.
3. `should_continue`: decides whether to retry or finish.

### Routing logic
- Score >= 80: finish.
- Attempts >= 3: finish as a safety limit.
- Otherwise: return to the generation node.

A cycle enables repetition; conditional routing controls when it stops.


In [6]:
builder = StateGraph(AgentState)

builder.add_node("generate", generate_answer)
builder.add_node("evaluate", evaluate_answer)

builder.add_edge(START, "generate")
builder.add_edge("generate", "evaluate")

builder.add_conditional_edges(
    "evaluate",
    should_continue,
    {
        "retry": "generate",
        "done": END
    }
)

graph = builder.compile()

In [7]:
result = graph.invoke(
    {
        "answer": "",
        "score": 0,
        "attempts": 0
    }
)

print("\nFinal result:")
print("Answer:", result["answer"])
print("Score:", result["score"])
print("Attempts:", result["attempts"])

Generation attempt: 1
Quality score: 40
Generation attempt: 2
Quality score: 70
Generation attempt: 3
Quality score: 90

Final result:
Answer: LangGraph is a framework for building stateful AI workflows with nodes, edges, and conditional routing.
Score: 90
Attempts: 3



### Expected output and interpretation

The example should produce three attempts with scores 40, 70, and 90.

The third attempt reaches the quality threshold, so the graph terminates successfully.

### Key terminology
- **Cycle:** A path that returns to an earlier node.
- **Iteration:** One repetition of the workflow.
- **Stopping condition:** A rule that ends repetition.
- **Termination:** The workflow reaches its end.
- **Recursion limit:** A safeguard against excessive graph execution.

### Key takeaway

Cycles make it possible to build iterative agents that improve their output instead of executing a fixed sequence only.

### Interview checkpoint
How would you prevent an agent from retrying forever?

**Answer:** Define a stopping condition and configure a recursion limit as an additional safeguard.
